# Sector & Commodity Returns and Seasonality

A research notebook in three parts:

1. **Recent returns**: S&P 500 sectors and commodities over a trailing window, as sorted bar charts.
2. **Next-quarter seasonality**: each name's average return in the upcoming calendar quarter over the last 5 years.
3. **Backtest**: would ranking names on that 5-year seasonal average have picked the following quarter's winners?

Prices come from Yahoo Finance via `yfinance`, so no API key is needed. Nothing is saved to disk unless you
set `SAVE_HTML = True` in the last section. Hover any bar for details.

In [ ]:
import pandas as pd
from IPython.display import display

import seasonality as sz

# --- settings ---------------------------------------------------------------------------
PERIOD = "1W"            # recent-returns window: 1D, 1W, 2W, 1M, 3M, 6M, 1Y, MTD, QTD, YTD
LOOKBACK_YEARS = 5       # years averaged for the seasonal charts and the backtest signal
TOP_N = 3                # names held in the top and bottom portfolios of the backtest
START = "2000-01-01"     # history to download; the backtest needs LOOKBACK_YEARS before it can start
TARGET_QUARTER = None    # None = the quarter after the latest price, or 1-4 to study any quarter

SECTORS = dict(sz.SECTORS)          # name -> Yahoo ticker; edit freely
COMMODITIES = dict(sz.COMMODITIES)  # front-month futures; dict(sz.COMMODITY_ETFS) for tradable ETFs

figures = []  # every chart is collected here for the optional HTML export

In [ ]:
sector_px = sz.load_prices(SECTORS, start=START)
commodity_px = sz.load_prices(COMMODITIES, start=START)

for label, px in [("Sectors", sector_px), ("Commodities", commodity_px)]:
    first = px.apply(lambda s: s.first_valid_index()).max()
    print(f"{label:<12} {px.shape[1]:>2} names · {px.index[0]:%Y-%m-%d} → {px.index[-1]:%Y-%m-%d}"
          f" · all names present from {first:%Y-%m-%d}")

## 1. Recent returns

Trailing return over `PERIOD`, from the close on or before the start of the window to the latest close.
Sector returns use the SPDR sector ETFs with dividends included; commodity returns use front-month futures.

In [ ]:
fig, sector_recent = sz.returns_chart(sector_px, SECTORS, PERIOD, "S&P Sector Returns")
figures.append(fig)
fig.show()

In [ ]:
fig, commodity_recent = sz.returns_chart(commodity_px, COMMODITIES, PERIOD, "Commodity Returns")
figures.append(fig)
fig.show()

In [ ]:
display(sz.styled(sector_recent))
display(sz.styled(commodity_recent))

## 2. Next-quarter seasonality

For the upcoming calendar quarter, each bar is the average return of that same quarter over the last
`LOOKBACK_YEARS` completed years. For example, in late September the chart shows the average Q4 return
over the last five Q4s. Hover a bar to see each year's return and how many were positive.

Five observations is a small sample: one outlier year can dominate the average. The median and hit rate in
the table, and the backtest in section 3, are the checks on whether the pattern is worth anything.

In [ ]:
target = sz.next_quarter(sector_px.index[-1]) if TARGET_QUARTER is None else TARGET_QUARTER
fig, sector_season = sz.seasonal_chart(sector_px, SECTORS, target, LOOKBACK_YEARS, "S&P Sector Returns")
figures.append(fig)
fig.show()
display(sz.styled(sector_season))

In [ ]:
fig, commodity_season = sz.seasonal_chart(commodity_px, COMMODITIES, target, LOOKBACK_YEARS, "Commodity Returns")
figures.append(fig)
fig.show()
display(sz.styled(commodity_season))

## 3. Backtest: does the 5-year seasonal average predict next quarter?

At the start of every quarter, rank the names by their average return in that same calendar quarter over the
previous `LOOKBACK_YEARS` years. That is exactly the number in the section 2 charts, computed only from data
available at the time. Then hold three portfolios for the quarter, rebalanced quarterly and equally weighted:

* **Top N**: the names with the highest seasonal average.
* **Bottom N**: the names with the lowest.
* **Equal weight**: every ranked name, the benchmark.

How to read the results:

* **Top − Bottom** is the spread. A positive average with a t-stat above about 2 suggests a real effect.
* **Rank IC** is the correlation between the seasonal ranking and the realised ranking, each quarter. Near 0
  means no predictive power. A mean around 0.05 or more with a t-stat above 2 is meaningful.
* **Sign hit rate** is how often the seasonal average had the same sign as the realised return. Compare it with
  **base rate positive**, the hit rate of always guessing "up".

No transaction costs, taxes or slippage are included.

In [ ]:
sector_bt = sz.backtest(sz.quarterly_returns(sector_px), years=LOOKBACK_YEARS, top_n=TOP_N)
first, last = sector_bt.returns.index[0], sector_bt.returns.index[-1]
print(f"S&P sectors: {len(sector_bt.returns)} quarters, {first} → {last}")
display(sz.styled(sector_bt.stats()))
display(sz.styled(sector_bt.ic_summary().to_frame("S&P sectors")))

In [ ]:
fig = sz.growth_chart(sector_bt, "S&P Sectors · Seasonal Backtest",
                     f"Growth of $1 · each quarter hold the {TOP_N} sectors with the highest average return "
                     f"in that quarter over the prior {LOOKBACK_YEARS} years")
figures.append(fig)
fig.show()
fig = sz.bar_chart(sector_bt.ic_by_quarter(), "S&P Sectors · Rank IC by Quarter",
                   "Mean correlation between the seasonal ranking and the realised ranking, by quarter predicted",
                   percent=False, decimals=2)
figures.append(fig)
fig.show()

In [ ]:
commodity_bt = sz.backtest(sz.quarterly_returns(commodity_px), years=LOOKBACK_YEARS, top_n=TOP_N)
first, last = commodity_bt.returns.index[0], commodity_bt.returns.index[-1]
print(f"Commodities: {len(commodity_bt.returns)} quarters, {first} → {last}")
display(sz.styled(commodity_bt.stats()))
display(sz.styled(commodity_bt.ic_summary().to_frame("Commodities")))

In [ ]:
fig = sz.growth_chart(commodity_bt, "Commodities · Seasonal Backtest",
                     f"Growth of $1 · each quarter hold the {TOP_N} commodities with the highest average return "
                     f"in that quarter over the prior {LOOKBACK_YEARS} years")
figures.append(fig)
fig.show()
fig = sz.bar_chart(commodity_bt.ic_by_quarter(), "Commodities · Rank IC by Quarter",
                   "Mean correlation between the seasonal ranking and the realised ranking, by quarter predicted",
                   percent=False, decimals=2)
figures.append(fig)
fig.show()

### What the backtest held recently

The last eight quarters of picks, with the realised return of each portfolio.

In [ ]:
for label, bt in [("S&P sectors", sector_bt), ("Commodities", commodity_bt)]:
    print(label)
    display(sz.styled(bt.holdings.join(bt.returns).tail(8)))

## 4. Save an HTML copy (optional)

Writes every chart above into one standalone page next to this notebook. It overwrites the same file each
time and does nothing unless `SAVE_HTML` is `True`.

In [ ]:
SAVE_HTML = False

if SAVE_HTML:
    path = sz.save_html("seasonality_report.html", figures)
    print(f"saved {path.resolve()}")

## Notes

* **Sectors** use the SPDR Select Sector ETFs. They track the S&P 500 sector indices closely but include fees.
  XLRE starts in 2015 and XLC in 2018, so they join the backtest once they have `LOOKBACK_YEARS` of history.
* **Commodity futures** on Yahoo are continuous front-month series. They jump at contract rolls, which adds noise
  to returns, especially for natural gas and crude oil. Switch to `sz.COMMODITY_ETFS` to backtest what an
  investor could actually hold; those include roll costs.
* **Sample size.** Each seasonal average rests on five numbers, and the backtest has roughly 80 quarters for
  sectors. Treat anything that is not consistent across the IC-by-quarter chart and both asset classes with
  suspicion.